# Step 2: Data Cleaning
Fix the dates, handle refunds (negative amounts), remove duplicates, and build a unique cardholder ID.

**Input:** `data/raw/pcard_2014.csv`  
**Output:** `data/processed/pcard_clean.csv`

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/pcard_2014.csv")
print(df.shape)

(442458, 11)


In [3]:
df.columns = ["year_month", "agency_number", "agency_name", "last_name",
              "first_initial", "description", "amount", "vendor",
              "transaction_date", "posted_date", "mcc"]

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 442458 entries, 0 to 442457
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   year_month        442458 non-null  int64  
 1   agency_number     442458 non-null  int64  
 2   agency_name       442458 non-null  str    
 3   last_name         442458 non-null  str    
 4   first_initial     442458 non-null  str    
 5   description       442458 non-null  str    
 6   amount            442458 non-null  float64
 7   vendor            442458 non-null  str    
 8   transaction_date  442458 non-null  str    
 9   posted_date       442458 non-null  str    
 10  mcc               442458 non-null  str    
dtypes: float64(1), int64(2), str(8)
memory usage: 37.1 MB


In [5]:
date_format = "%m/%d/%Y %I:%M:%S %p"
df["transaction_date"] = pd.to_datetime(df["transaction_date"], format = date_format)
df["posted_date"] = pd.to_datetime(df["posted_date"], format = date_format)

In [6]:
text_cols = ["agency_name", "last_name", "first_initial", "description", "vendor", "mcc"]
for col in text_cols:
    df[col] = (df[col].astype(str)
                      .str.strip()
                      .str.upper()
                      .str.replace(r"\s+", " ", regex=True))

In [7]:
main_name = df.groupby("agency_number")["agency_name"].agg(lambda s: s.value_counts().index[0])
df["agency_name"] = df["agency_number"].map(main_name)
print("Agency names:", df["agency_name"].nunique())

Agency names: 116


In [8]:
df["cardholder_id"] = (df["agency_number"].astype(str) + "_" + df["last_name"] + "_" + df["first_initial"])
print(df["cardholder_id"].nunique())

5448


In [9]:
refunds = df[df["amount"] <= 0]
df = df[df["amount"] > 0].copy()
print("refunds removed: ", len(refunds))

refunds removed:  14531


In [10]:
is_account_entry = df["last_name"].str.contains(r"\d", na=False)
print("Account entries removed", is_account_entry.sum())
df = df[~is_account_entry].copy()

Account entries removed 55


In [11]:
print("Exat dupes: ", df.duplicated().sum())

Exat dupes:  14330


In [14]:
df = df.sort_values(["cardholder_id", "transaction_date"]).reset_index(drop=True)
df.to_csv("../data/processed/pcard_clean.csv", index=False)
print("Saved:", df.shape)

PermissionError: [Errno 13] Permission denied: '../data/processed/pcard_clean.csv'

In [ ]:
purchases_per_cardholder = df.groupby("cardholder_id").size()

print(purchases_per_cardholder.describe())

few = purchases_per_cardholder[purchases_per_cardholder < 5]
print("Cardholders with < 5 purchases:", len(few),
      f"({len(few) / len(purchases_per_cardholder):.1%})")

# The ones with the fewest
print(purchases_per_cardholder.sort_values().head(10))

count    5425.000000
mean       78.870415
std       227.434185
min         1.000000
25%        12.000000
50%        33.000000
75%        85.000000
max      9650.000000
dtype: float64
Cardholders with < 5 purchases: 650 (12.0%)
cardholder_id
76000_EPPLER_C     1
1000_ALLEN_N       1
76000_FELLOWS_S    1
1000_AMBROSE_E     1
1000_AMBARDAR_M    1
76000_FAGAN_J      1
30800_SAULS_D      1
76000_BUTLER_E     1
1000_EMERY_P       1
34500_HOLT_R       1
dtype: int64


In [ ]:
df["posting_delay"] = (df["posted_date"] - df["transaction_date"]).dt.days

print(df["posted_date"].describe())

late = df[df["posting_delay"] > 60]
print("Posted after 60+ days:", len(late))
print(late.sort_values("posting_delay", ascending=False)
          [["cardholder_id", "vendor", "amount", "transaction_date",
            "posted_date", "posting_delay"]])

count                        427872
mean     2013-12-30 07:05:44.693739
min             2013-07-01 00:00:00
25%             2013-09-26 00:00:00
50%             2014-01-07 00:00:00
75%             2014-04-03 00:00:00
max             2014-06-30 00:00:00
Name: posted_date, dtype: object
Posted after 60+ days: 4
            cardholder_id                vendor   amount transaction_date  \
205206      47700_GOMEZ_K   FSA HOSPITALITY LLC   1540.0       2014-01-16   
82798       1000_ROPERS_A  BETHESDA COURT HOTEL  15120.0       2013-05-16   
135692     16000_LOFTON_L  LITTLE AMERICA HOTEL    432.4       2013-09-18   
410846  9000_ESTABROOKS_C        SEQUOYAH LODGE     77.0       2013-08-01   

       posted_date  posting_delay  
205206  2014-04-14             88  
82798   2013-07-24             69  
135692  2013-11-25             68  
410846  2013-10-02             62  


In [ ]:
#rows = 0 because i cleaned it above 
# Rows where the "last name" contains a digit
numeric_cards = df[df["last_name"].str.contains(r"\d", na=False)]

print("Rows:", len(numeric_cards))
print(numeric_cards.groupby("cardholder_id").size().sort_values(ascending=False))
print(numeric_cards["vendor"].value_counts())

Rows: 0
Series([], dtype: int64)
Series([], Name: count, dtype: int64)


In [ ]:
df.head()

,year_month,agency_number,agency_name,last_name,first_initial,description,amount,vendor,transaction_date,posted_date,mcc,cardholder_id,posting_delay
0,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,2.80,PIKEPASS REBILLS,2013-07-01,2013-07-02,TOLLS AND BRIDGE FEES,1000_ABBOTT_T,1
1,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,AIR TRAVEL,297.60,DELTA AIR 0062335020780,2013-07-05,2013-07-08,DELTA,1000_ABBOTT_T,3
2,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,068934075 PCS,362.37,XEROX CORPORATION/RBO,2013-07-12,2013-07-15,"OFFICE, PHOTOGRAPHIC, PHOTOCOPY, AND MICROFILM...",1000_ABBOTT_T,3
3,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,1068.55,BUSINESS IMAGING SYSTEMS,2013-07-12,2013-07-15,"OFFICE, PHOTOGRAPHIC, PHOTOCOPY, AND MICROFILM...",1000_ABBOTT_T,3
4,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,285.00,OK DEPT OF VO-TECH ED,2013-07-12,2013-07-15,GOVERNMENT SERVICES--NOT ELSEWHERE CLASSIFIED,1000_ABBOTT_T,3


In [ ]:
round(df.describe(),2)

,year_month,agency_number,amount,transaction_date,posted_date,posting_delay
count,427872.00,427872.00,427872.00,427872,427872,427872.00
mean,201357.27,42784.05,438.30,2013-12-28 11:05:29.750954,2013-12-30 07:05:44.693739,1.83
min,201307.00,1000.00,0.01,2013-05-16 00:00:00,2013-07-01 00:00:00,0.00
25%,201309.00,1000.00,35.90,2013-09-25 00:00:00,2013-09-26 00:00:00,1.00
50%,201401.00,47700.00,113.35,2014-01-06 00:00:00,2014-01-07 00:00:00,1.00
75%,201404.00,76000.00,359.02,2014-04-01 00:00:00,2014-04-03 00:00:00,3.00
max,201406.00,98000.00,1089180.00,2014-06-30 00:00:00,2014-06-30 00:00:00,88.00
std,47.11,33381.27,3567.97,NaN,NaN,1.27


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 427872 entries, 0 to 427871
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   year_month        427872 non-null  int64         
 1   agency_number     427872 non-null  int64         
 2   agency_name       427872 non-null  str           
 3   last_name         427872 non-null  str           
 4   first_initial     427872 non-null  str           
 5   description       427872 non-null  str           
 6   amount            427872 non-null  float64       
 7   vendor            427872 non-null  str           
 8   transaction_date  427872 non-null  datetime64[us]
 9   posted_date       427872 non-null  datetime64[us]
 10  mcc               427872 non-null  str           
 11  cardholder_id     427872 non-null  str           
dtypes: datetime64[us](2), float64(1), int64(2), str(7)
memory usage: 39.2 MB
